# Roof + circulation

Paths people walk on, over any roof. The module takes drawn path centre lines and returns a
`Contribution`: solid to **add** (the walking surface, with ramps and stairs), space to **cut**
(headroom over it), and solid to **keep** (under the walking surface, safe from later processes).
`combine` puts it on a host. Here the host is a plain stand-in slab; later it is the shared roof,
with the other modules' contributions alongside this one.

Each step below is one group of components on a Grasshopper canvas, written as code. The logic lives in
`roof_circulation/`, where it is tested. Units are millimetres.

In [ ]:
%matplotlib inline
import roof_circulation as work

## Step 1 · The brief
*Canvas: number sliders*

The brief holds what is **controlled**: the rules every path keeps, whatever is drawn. A ramp is never
steeper than `ramp_max`; steeper runs become stairs, with no riser over `riser`, and a landing `width / 2`
long at each end of a flight. `headroom` is kept empty over the path, and `support` under it is protected.
`length`, `span` and `thickness` only size the stand-in roof.

In [ ]:
brief = work.Brief(width=1200, ramp_max=1 / 12, riser=150, going_min=280, headroom=2100, support=300)

## Step 2 · The drawing
*Canvas: referenced polylines*

The drawing holds what is **free**: where the paths go, and the walking level at each vertex (its Z).
Ramps are blue, stairs red, with a tick at each riser.

**Try:** move the stairs' top vertex closer, `[7500, 3500, 1400]`, and see Step 4 flag the steps as too shallow.

In [ ]:
drawing = work.Drawing.sample()
runs = work.steps.segments(brief, drawing)
for seg in runs[0]:
    print(seg.kind, "run", round(seg.run), "rise", round(seg.rise), "risers", seg.risers, "going", round(seg.going))
bounds = work.steps.extent(brief, drawing)
work.preview.plan(bounds, runs);

## Step 3 · The module, on a host
*Canvas: **Volumetric Box**, **Volumetric Boolean***

`circulation` is the whole module. `combine` adds every module's solid to the host, then cuts every
module's space, so the headroom stays clear whatever else is added.

In [ ]:
part = work.steps.circulation(brief, drawing)
host = work.steps.host(brief)
solid = work.combine(host, [part])
work.preview.section(bounds, solid, axis="y", at=1500, title="along the ramp")
work.preview.section(bounds, solid, axis="x", at=7500, title="up the stairs");

## Step 4 · Checks
Every rule measured on the finished solid: floor underfoot and nothing overhead at every sample, and no
stairs too shallow for their rise. Run this again after other modules are combined.

In [ ]:
print(work.checks.path_clear(brief, runs, solid))
print("steps too shallow:", work.checks.steep_stairs(brief, runs))

## Step 5 · Export
*Canvas: **Volumetric Mesh***

Writes the mesh (`.stl`) and the field (`.json`) to `out/`, which git ignores. Lower `voxel` for a finer mesh.

In [ ]:
vertices, faces = work.steps.export(brief, solid, bounds, "out/roof_circulation")
print("{:,} triangles".format(len(faces)))